# 07 — Zonal LMP Choropleth Map

**Purpose:** Visualise E4ST zonal locational marginal prices (LMPs) as a
Folium choropleth over BA-territory polygons for all completed scenarios.

LMPs emerge from the zone-level power-balance shadow price in the copper-plate
model — there is no nodal separation or congestion premium.  All zones within
a BA share a single price.

**Inputs:**
- `data/processed/e4st_results/{scenario}/lmp.parquet` — one row per BA zone
- `data/processed/ba_territories.geojson` — BA polygon geometries
- `data/processed/network_metadata.json` — scenario registry

**Outputs:**
- `data/processed/lmp_map.html` — interactive Folium map (three scenario layers)

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import geopandas as gpd
import folium
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results'
BA_GEO_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson'
META_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'network_metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

# Only use OPTIMAL scenarios
SCENARIOS = [
    s['name']
    for s in meta['scenarios_completed']
    if s['status'] == 'OPTIMAL'
]

SCENARIO_LABELS = {
    'baseline':      'Baseline',
    'carbon_tax_50': 'Carbon Tax $50/tCO₂',
    'ces_achievable': 'CES 50 % (max feasible)',
}

print('Scenarios to map:', SCENARIOS)

In [ ]:
# ── Load LMP data for all scenarios ───────────────────────────────────────────
lmp_frames = {}
for sc in SCENARIOS:
    df = pd.read_parquet(RESULTS_DIR / sc / 'lmp.parquet')
    lmp_frames[sc] = df
    print(f'{sc:20s}: {len(df):3d} zones,  LMP range '
          f'${df["lmp_mwh"].min():.2f}–${df["lmp_mwh"].max():.2f}/MWh')

# ── Load BA territory geometries ───────────────────────────────────────────────
ba_gdf = gpd.read_file(BA_GEO_PATH)[['ba_code', 'ba_name', 'geometry']]
print(f'\nBA territories loaded: {len(ba_gdf)} polygons')

In [ ]:
# ── Merge each scenario's LMPs onto the BA geometries ─────────────────────────
# Only keep BAs that appear in the model (inner join → 67 zones)
merged = {}
for sc, df in lmp_frames.items():
    gdf = ba_gdf.merge(df[['ba', 'lmp_mwh']], left_on='ba_code', right_on='ba', how='inner')
    merged[sc] = gdf
    print(f'{sc:20s}: {len(gdf)} zones matched to geometry')

# Shared color scale across ALL scenarios so layers are visually comparable
all_lmps = pd.concat([df['lmp_mwh'] for df in lmp_frames.values()])
LMP_VMIN = all_lmps.quantile(0.02)
LMP_VMAX = all_lmps.quantile(0.98)
print(f'\nShared color scale: ${LMP_VMIN:.2f} – ${LMP_VMAX:.2f}/MWh')

In [ ]:
# ── Color helper ──────────────────────────────────────────────────────────────
CMAP = cm.get_cmap('YlOrRd')
NORM = mcolors.Normalize(vmin=LMP_VMIN, vmax=LMP_VMAX)

def lmp_hex(value: float) -> str:
    """Map an LMP value to a hex colour string using the shared scale."""
    rgba = CMAP(NORM(value))
    return mcolors.to_hex(rgba)

# Quick sanity
print('Low  LMP colour:', lmp_hex(LMP_VMIN))
print('High LMP colour:', lmp_hex(LMP_VMAX))

In [ ]:
# ── Build Folium map with one FeatureGroup per scenario ────────────────────────
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

for sc in SCENARIOS:
    gdf = merged[sc]
    label = SCENARIO_LABELS.get(sc, sc)
    fg = folium.FeatureGroup(name=label, show=(sc == 'baseline'))

    for _, row in gdf.iterrows():
        lmp_val = row['lmp_mwh']
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda _, v=lmp_val: {
                'fillColor': lmp_hex(v),
                'color': '#555555',
                'weight': 0.5,
                'fillOpacity': 0.75,
            },
            tooltip=folium.Tooltip(
                f"<b>{row['ba_code']}</b> — {row['ba_name']}<br>"
                f"LMP: <b>${lmp_val:.2f}/MWh</b>",
                sticky=True,
            ),
        ).add_to(fg)

    fg.add_to(m)

# ── Colorbar legend (static PNG injected as HTML) ─────────────────────────────
fig_cb, ax_cb = plt.subplots(figsize=(4, 0.35))
fig_cb.subplots_adjust(left=0.05, right=0.95, top=1, bottom=0)
cb = matplotlib.colorbar.ColorbarBase(
    ax_cb, cmap=CMAP, norm=NORM, orientation='horizontal'
)
cb.set_label('LMP ($/MWh)', fontsize=8)
cb.ax.tick_params(labelsize=7)

import io, base64
buf = io.BytesIO()
fig_cb.savefig(buf, format='png', bbox_inches='tight', dpi=120,
               facecolor='white', transparent=False)
plt.close(fig_cb)
buf.seek(0)
img_b64 = base64.b64encode(buf.read()).decode()

legend_html = f"""
<div style="position:fixed; bottom:30px; left:30px; z-index:9999;
            background:white; padding:8px 10px; border-radius:6px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-family:sans-serif;">
  <b style="font-size:12px;">Zonal LMP</b><br>
  <img src="data:image/png;base64,{img_b64}" width="220">
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

out_path = PROJECT_ROOT / 'data' / 'processed' / 'lmp_map.html'
m.save(str(out_path))
print(f'Saved → {out_path}')
m

In [ ]:
# ── Quick numeric summary ─────────────────────────────────────────────────────
rows = []
for sc in SCENARIOS:
    df = lmp_frames[sc]
    rows.append({
        'Scenario': SCENARIO_LABELS.get(sc, sc),
        'Mean LMP ($/MWh)':   df['lmp_mwh'].mean(),
        'Min LMP ($/MWh)':    df['lmp_mwh'].min(),
        'Max LMP ($/MWh)':    df['lmp_mwh'].max(),
        'Std Dev ($/MWh)':    df['lmp_mwh'].std(),
    })

summary = pd.DataFrame(rows).set_index('Scenario')
summary.style.format('{:.2f}')